<a href="https://colab.research.google.com/github/sahilkavishka/-EdTech-Analytics/blob/main/data_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# 1. Install required packages including 'isodate' for advanced time parsing
!pip install google-api-python-client pandas isodate

In [3]:
import os
import time
import logging
import pandas as pd
import isodate
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from typing import List, Dict, Any

In [7]:
# 2. Configure Professional Logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s: %(message)s')

# --- CONFIGURATION ---
API_KEY = 'YOUR_API_KEY_HERE'  # Insert your Google Cloud API Key
MAX_RESULTS_PER_QUERY = 300

SEARCH_QUERIES = [
    'Sri Lanka A/L Science Maths lessons',
    'Sri Lanka A/L Commerce Arts classes',
    'Sri Lanka O/L Mathematics Science Past Papers',
    'Sri Lanka education exam motivation',
    'Sri Lanka O/L ICT History revision'
]

def categorize_content(title: str) -> str:
    """Auto-categorizes videos based on title keywords (Rule-based NLP)."""
    title_upper = title.upper()
    if 'A/L' in title_upper or 'ADVANCED LEVEL' in title_upper or 'AL ' in title_upper:
        return 'A/L Content'
    elif 'O/L' in title_upper or 'ORDINARY LEVEL' in title_upper or 'OL ' in title_upper:
        return 'O/L Content'
    elif 'MOTIVATION' in title_upper or 'TIPS' in title_upper or 'STUDY' in title_upper:
        return 'Motivation & Tips'
    elif 'PAPER' in title_upper or 'PAST' in title_upper or 'MCQ' in title_upper:
        return 'Past Papers & Revision'
    else:
        return 'General Education'

def fetch_ultimate_market_data(api_key: str, queries: List[str], max_per_query: int) -> pd.DataFrame:
    youtube = build('youtube', 'v3', developerKey=api_key)
    all_video_ids = set()

    print("🚀 Phase 1: Harvesting Video IDs from Market Segments...")
    for query in tqdm(queries, desc="Querying Segments"):
        next_page_token = None
        fetched = 0
        while fetched < max_per_query:
            try:
                search_request = youtube.search().list(
                    q=query, part='id', type='video',
                    maxResults=min(50, max_per_query - fetched),
                    pageToken=next_page_token, regionCode='LK'
                )
                response = search_request.execute()
                new_ids = [item['id']['videoId'] for item in response.get('items', [])]
                all_video_ids.update(new_ids)
                fetched += len(new_ids)
                next_page_token = response.get('nextPageToken')
                if not next_page_token: break
            except HttpError as e:
                logging.error(f"Error fetching IDs: {e}")
                break

    video_ids_list = list(all_video_ids)
    if not video_ids_list:
        return pd.DataFrame()

    print(f"✅ Found {len(video_ids_list)} unique videos. \n🚀 Phase 2: Deep Extraction of Video Metrics...")

    video_data = []
    channel_ids = set()

    for i in tqdm(range(0, len(video_ids_list), 50), desc="Fetching Video Stats"):
        batch_ids = video_ids_list[i:i+50]
        try:
            stats_request = youtube.videos().list(
                id=','.join(batch_ids),
                part='snippet,statistics,contentDetails'
            )
            response = stats_request.execute()

            for video in response.get('items', []):
                snip = video['snippet']
                stat = video.get('statistics', {})
                cont = video.get('contentDetails', {})

                channel_ids.add(snip['channelId'])

                try:
                    dur_str = cont.get('duration', 'PT0S')
                    dur_sec = int(isodate.parse_duration(dur_str).total_seconds())
                except:
                    dur_sec = 0

                pub_date = pd.to_datetime(snip.get('publishedAt', ''))

                video_data.append({
                    'Video_ID': video['id'],
                    'Channel_ID': snip['channelId'],
                    'Channel_Name': snip.get('channelTitle', 'Unknown'),
                    'Title': snip.get('title', ''),
                    'Content_Category': categorize_content(snip.get('title', '')),
                    'Published_Date': pub_date.date() if pd.notnull(pub_date) else None,
                    'Upload_Hour': pub_date.hour if pd.notnull(pub_date) else None,
                    'Duration_Seconds': dur_sec,
                    'Has_Captions': cont.get('caption', 'false').capitalize(),
                    'Tags_Count': len(snip.get('tags', [])),
                    'Views': int(stat.get('viewCount', 0)),
                    'Likes': int(stat.get('likeCount', 0)),
                    'Comments': int(stat.get('commentCount', 0))
                })
        except Exception as e:
            continue

    print(f"🚀 Phase 3: Analyzing Channel Authority for {len(channel_ids)} channels...")
    channel_stats = {}
    channel_ids_list = list(channel_ids)

    for i in tqdm(range(0, len(channel_ids_list), 50), desc="Fetching Channel Stats"):
        batch_cids = channel_ids_list[i:i+50]
        try:
            ch_request = youtube.channels().list(
                id=','.join(batch_cids),
                part='statistics'
            )
            ch_response = ch_request.execute()
            for ch in ch_response.get('items', []):
                channel_stats[ch['id']] = int(ch['statistics'].get('subscriberCount', 0))
        except:
            continue

    # Phase 4: Data Merging & Super Feature Engineering
    print("🚀 Phase 4: Compiling Final Super Dataset...")
    df = pd.DataFrame(video_data)

    if not df.empty:
        # Map subscriber counts
        df['Channel_Subscribers'] = df['Channel_ID'].map(channel_stats).fillna(0)

        # Calculate Age in Days
        today = pd.to_datetime('today').date()
        df['Age_in_Days'] = (today - df['Published_Date']).dt.days
        df['Age_in_Days'] = df['Age_in_Days'].apply(lambda x: 1 if x <= 0 else x) # Prevent division by zero

        # Super Metrics
        df['Engagement_Rate_%'] = ((df['Likes'] + df['Comments']) / df['Views'] * 100).fillna(0).round(2)
        df['Views_Per_Day'] = (df['Views'] / df['Age_in_Days']).round(2)

        # Viral Multiplier: How many times views exceeded subscriber count
        df['Viral_Multiplier'] = (df['Views'] / df['Channel_Subscribers'].replace(0, 1)).round(2)

        # Clean up
        df = df[df['Views'] > 1000] # Focus on significant data
        df = df.drop(columns=['Channel_ID'])
        df = df.sort_values(by='Views_Per_Day', ascending=False).reset_index(drop=True)

    return df

# Execute Pipeline
final_df = fetch_ultimate_market_data(API_KEY, SEARCH_QUERIES, MAX_RESULTS_PER_QUERY)

if not final_df.empty:
    csv_name = 'Ultimate_EdTech_Intelligence_2026.csv'
    final_df.to_csv(csv_name, index=False)
    print(f"\n✅ SUCCESS! Enterprise Dataset Generated: {csv_name}")
    print(f"Total Quality Records: {len(final_df)}")
    display(final_df[['Channel_Name', 'Content_Category', 'Views_Per_Day', 'Viral_Multiplier', 'Engagement_Rate_%']].head())
else:
    print("\n[Notice]: Dataset empty. Check API quota.")


--- Production Dataset Saved: Advanced_EdTech_Analytics_Dataset.csv ---
Total Records Fetched: 200


,Title,Views,Engagement_Rate_%
0,Hydrophobic Club Moss Spores,80570520,3.04
1,Hydrophobic Club Moss Spores,80570520,3.04
2,Luca(faster) #superclever #abacus #franchise #...,73378472,2.20
3,Rugby training with the World Champion Springb...,66969521,0.59
4,using the correct braking method can get you o...,39404430,0.72
